# Optimización de Hiperparámetros

## Pasos Preliminares

### Importación del Pipeline de Preprocesamiento

El pipeline de preprocesamiento desarrollado en notebooks anteriores se importa desde el módulo compartido [`utils/housing_preprocessing.py`](utils/housing_preprocessing.py). Usamos un valor por defecto bajo para `n_clusters` ya que vamos a ajustar este hiperparámetro.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from utils.housing_preprocessing import get_preprocessing_pipeline
preprocessing = get_preprocessing_pipeline(n_clusters=10)  # Valor bajo por defecto, será ajustado

In [ ]:
full_pipeline = Pipeline([    ("preprocessing", preprocessing),    ("random_forest", RandomForestRegressor(random_state=42, n_jobs=1)),])

### Carga de Datos

La carga de datos con la división estratificada entrenamiento/prueba se importa desde [`utils/load_california.py`](utils/load_california.py).

In [ ]:
from utils.load_california import load_housing_dataX_train, X_test, y_train, y_test = load_housing_data()

## Hiperparámetros Relevantes

Para el pipeline de preprocesamiento:

| Hiperparámetro      | Descripción                                                 |
|---------------------|-------------------------------------------------------------|
| `n_clusters`        | Número de clústeres correspondientes a zonas geográficas.   |
| `gamma`             | Tasa de decaimiento de la similitud con el centroide.       |
| `strategy`          | Estrategia de imputación para valores faltantes (por defecto es la media). |

Para RandomForestRegressor:

| Hiperparámetro      | Descripción |
|---------------------|-------------|
| `n_estimators`     | Número de árboles en el bosque. Más árboles pueden mejorar la precisión pero aumentan el tiempo de cómputo. |
| `max_depth`        | Profundidad máxima de cada árbol. Un valor bajo puede llevar a *underfitting*, mientras que un valor alto puede llevar a *overfitting*. |
| `max_features`     | Número de *características* consideradas en cada división. Puede ser un entero, un porcentaje, `"sqrt"` o `"log2"`. Menos *características* pueden reducir la varianza (y por tanto el *overfitting*). |
| `min_samples_split` | Número mínimo de muestras necesarias para dividir un nodo. Valores más altos reducen el *overfitting*. |
| `min_samples_leaf`  | Número mínimo de muestras en una hoja. Valores más altos suavizan la predicción. |
| `max_samples`      | Porcentaje de muestras usadas en cada árbol. Útil para reducir el *overfitting*. |

## Ajuste de Hiperparámetros

### 1ª Iteración

Empecemos con una búsqueda aleatoria preliminar usando un rango amplio de valores.

In [ ]:
%%time
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import randint, uniform

param_dist = {
    'preprocessing__geo__n_clusters': randint(low=3, high=200),
    'random_forest__n_estimators': randint(100, 500),  # Cualquier entero entre 100 y 499
    'random_forest__max_depth': randint(10, 110),      # Cualquier entero entre 10 y 109
    'random_forest__min_samples_split': randint(2, 20),
    'random_forest__min_samples_leaf': randint(1, 20),
    'random_forest__max_features': ['sqrt', 'log2', None]
}

rnd_search = RandomizedSearchCV(
    estimator = full_pipeline, 
    param_distributions=param_dist, 
    n_iter=40, 
    cv=5,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1   # Usar todos los núcleos de CPU en paralelo
    )
_ = rnd_search.fit(X_train, y_train)

`%%time` es un [comando mágico de Jupyter](https://ipython.readthedocs.io/en/stable/interactive/magics.html#magic-time) que mide el tiempo de ejecución de la celda.

Podemos ver los resultados de los mejores modelos encontrados:

In [ ]:
cv_res = pd.DataFrame(rnd_search.cv_results_)cv_res.sort_values(by="mean_test_score", ascending=False, inplace=True)cv_res = cv_res[['param_preprocessing__geo__n_clusters',                 'param_random_forest__n_estimators',                 'param_random_forest__max_depth',                 'param_random_forest__min_samples_split',                 'param_random_forest__min_samples_leaf',                 'param_random_forest__max_features',                 "mean_test_score"]]cv_res.columns = ["n_clusters", "n_estimators", "max_depth", "min_samples_split", "min_samples_leaf", "max_features", "mean_test_score"]cv_res["mean_test_score"] = -cv_res["mean_test_score"].round().astype(np.int64)cv_res.head()

Ahora podemos realizar iteraciones sucesivas fijando las *características* donde todos los mejores resultados han convergido a un único valor, y definiendo un diccionario más estrecho de valores a probar centrado en los mejores resultados.

### 2ª Iteración

In [ ]:
%%time
full_pipeline.set_params(random_forest__max_features="sqrt") # Fijar el valor de max_features, que ha convergido a "sqrt"
param_dist = {
    'preprocessing__geo__n_clusters': randint(low=55, high=150),
    'random_forest__n_estimators': randint(200, 300),
    'random_forest__max_depth': randint(44, 97),
    'random_forest__min_samples_split': randint(2, 14),
    'random_forest__min_samples_leaf': randint(1, 5),
}
rnd_search = RandomizedSearchCV(
    estimator = full_pipeline, 
    param_distributions=param_dist, 
    n_iter=40, 
    cv=5,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1   # Usar todos los núcleos de CPU en paralelo
    )
_ = rnd_search.fit(X_train, y_train)

In [ ]:
cv_res = pd.DataFrame(rnd_search.cv_results_)cv_res.sort_values(by="mean_test_score", ascending=False, inplace=True)cv_res = cv_res[['param_preprocessing__geo__n_clusters',                 'param_random_forest__n_estimators',                 'param_random_forest__max_depth',                 'param_random_forest__min_samples_split',                 'param_random_forest__min_samples_leaf',                 "mean_test_score"]]cv_res.columns = ["n_clusters", "n_estimators", "max_depth", "min_samples_split", "min_samples_leaf", "mean_test_score"]cv_res["mean_test_score"] = -cv_res["mean_test_score"].round().astype(np.int64)cv_res.head()